# P1-E16.7G Renamed(P1-E11.G) Four-Endpoint Statistical, Reproducibility and Publication Lock

**Purpose.** 
Consolidate the authenticated outputs of P1-E16.7A–F into one manuscript-facing evidence lock. This notebook performs **no model fitting, hyperparameter selection, threshold tuning, refitting, or recalibration**.

The four endpoints are fixed as **Normal vs AD**, **MCI vs AD**, **Normal vs MCI**, and **Normal/MCI/AD three-class classification**. Evidence is kept in three non-interchangeable layers: nested development out-of-fold (OOF), retrospective locked official test, and frozen external evaluation.

## 1. Configuration

Set only `EXPERIMENT_ROOT` if the project is stored elsewhere. Each upstream directory and table is explicit; ambiguous recursive discovery is prohibited.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

EXPERIMENT_ROOT = Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT = EXPERIMENT_ROOT/'paper1_outputs'
OUTPUT_ROOT = OUTPUTS_ROOT/'p1_e16_7g_final_four_endpoint_publication_lock'
TABLES = OUTPUT_ROOT/'tables'; FIGURES = OUTPUT_ROOT/'figures'; LOGS = OUTPUT_ROOT/'logs'
for folder in (OUTPUT_ROOT, TABLES, FIGURES, LOGS): folder.mkdir(parents=True, exist_ok=True)

EXPECTED_ENDPOINTS = ['normal_vs_ad','mci_vs_ad','normal_vs_mci','three_class']
ENDPOINT_LABELS = {'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD',
                   'normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
CHANCE_LEVEL = {'normal_vs_ad':0.5,'mci_vs_ad':0.5,'normal_vs_mci':0.5,'three_class':1/3}
SEED = 20261012

ROOTS = {
 '7A': OUTPUTS_ROOT/'p1_e16_7a_four_endpoint_nested_conventional_freeze',
 '7BR': OUTPUTS_ROOT/'p1_e16_7b_r_unified_four_endpoint_nested_method_comparison',
 '7C': OUTPUTS_ROOT/'p1_e16_7c_endpoint_selection_cross_fitted_reduced_fusion_freeze',
 '7D': OUTPUTS_ROOT/'p1_e16_7d_full_development_refit_retrospective_locked_test',
 '7D2': OUTPUTS_ROOT/'p1_e16_7d_2_matched_official_test_comparison',
 '7D3': OUTPUTS_ROOT/'p1_e16_7d_3_representation_classifier_fairness_audit',
 '7E': OUTPUTS_ROOT/'p1_e16_7e_frozen_external_reevaluation_transportability_shift_chronology',
 '7F': OUTPUTS_ROOT/'p1_e16_7f_four_endpoint_age_confounding_normative_sensitivity',
}

FILES = {
 '7A_integrity': ROOTS['7A']/'tables'/'15_FINAL_INTEGRITY_CHECKS.csv',
 '7A_development': ROOTS['7A']/'tables'/'08_NESTED_DEVELOPMENT_METRICS.csv',
 '7BR_integrity': ROOTS['7BR']/'tables'/'12_FINAL_INTEGRITY_CHECKS.csv',
 '7BR_summary': ROOTS['7BR']/'tables'/'09_METHOD_PERFORMANCE_SUMMARY.csv',
 '7BR_predictions': ROOTS['7BR']/'tables'/'08_ALL_METHOD_NESTED_OOF_PREDICTIONS.csv',
 '7C_integrity': ROOTS['7C']/'tables'/'08_FINAL_INTEGRITY_CHECKS.csv',
 '7C_selection': ROOTS['7C']/'tables'/'06_ENDPOINT_SELECTION_DECISIONS.csv',
 '7D_integrity': ROOTS['7D']/'tables'/'09_FINAL_INTEGRITY_CHECKS.csv',
 '7D_test': ROOTS['7D']/'tables'/'07_RETROSPECTIVE_LOCKED_TEST_METRICS.csv',
 '7D2_integrity': ROOTS['7D2']/'tables'/'07_FINAL_INTEGRITY_CHECKS.csv',
 '7D2_metrics': ROOTS['7D2']/'tables'/'04_IDENTICALLY_COMPUTED_MATCHED_METRICS.csv',
 '7D2_paired': ROOTS['7D2']/'tables'/'05_PAIRED_BALANCED_ACCURACY_DIFFERENCES.csv',
 '7D3_integrity': ROOTS['7D3']/'tables'/'08_FINAL_INTEGRITY_CHECKS.csv',
 '7D3_metrics': ROOTS['7D3']/'tables'/'06_CONTROLLED_BLOCK_PERFORMANCE.csv',
 '7D3_paired': ROOTS['7D3']/'tables'/'07_WITHIN_BLOCK_PAIRED_COMPARISONS.csv',
 '7E_integrity': ROOTS['7E']/'tables'/'13_FINAL_INTEGRITY_CHECKS.csv',
 '7E_metrics': ROOTS['7E']/'tables'/'08_EXTERNAL_METRICS.csv',
 '7E_intervals': ROOTS['7E']/'tables'/'09_PARTICIPANT_BOOTSTRAP_INTERVALS.csv',
 '7E_paired': ROOTS['7E']/'tables'/'10_PAIRED_EXTERNAL_METHOD_DIFFERENCES.csv',
 '7E_shift': ROOTS['7E']/'tables'/'11_FEATURE_SHIFT_STANDARDIZED_MEAN_DIFFERENCES.csv',
 '7F_integrity': ROOTS['7F']/'tables'/'12_FINAL_INTEGRITY_CHECKS.csv',
 '7F_summary': ROOTS['7F']/'tables'/'08_PERFORMANCE_SUMMARY_WITH_95CI.csv',
 '7F_deltas': ROOTS['7F']/'tables'/'09_PAIRED_AGE_SENSITIVITY_DELTAS.csv',
 '7F_age': ROOTS['7F']/'tables'/'10_AGE_DISTRIBUTION_BY_LABEL.csv',
}

def sha256_file(path, chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(chunk),b''): h.update(block)
    return h.hexdigest()

def save_table(frame,name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False); return path

def find_col(frame, candidates, required=True):
    lookup={str(c).lower():c for c in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:return lookup[candidate.lower()]
    if required: raise KeyError(f'None of {candidates} found in {list(frame.columns)}')
    return None

sns.set_theme(style='whitegrid',context='paper',font_scale=1.1)
print('Output:',OUTPUT_ROOT)

## 2. Authenticate the upstream evidence

The notebook stops if any mandatory source is missing. Optional shift results may be absent only when the upstream notebook explicitly produced no shift table.

In [ ]:
mandatory=[p for key,p in FILES.items() if key!='7E_shift']
missing=[str(p) for p in mandatory if not p.is_file()]
if missing: raise FileNotFoundError('Missing required final source file(s):\n'+'\n'.join(missing))

inventory=[]
for role,path in FILES.items():
    inventory.append({'role':role,'path':str(path),'exists':path.is_file(),
                      'sha256':sha256_file(path) if path.is_file() else None,
                      'size_bytes':path.stat().st_size if path.is_file() else None})
inventory=pd.DataFrame(inventory)
save_table(inventory,'01_AUTHENTICATED_INPUT_INVENTORY')
display(inventory)
assert inventory.loc[inventory.role!='7E_shift','exists'].all()
print('All mandatory P1-E16.7A–F source tables authenticated.')

## 3. Verify upstream integrity locks

Each final upstream integrity table must contain only passing checks. Different notebooks used slightly different names for the Boolean result column, so it is resolved transparently.

In [ ]:
integrity_rows=[]
for stage in ['7A','7BR','7C','7D','7D2','7D3','7E','7F']:
    frame=pd.read_csv(FILES[f'{stage}_integrity'])
    pass_col=find_col(frame,['pass','passed','value'])
    values=frame[pass_col]
    if values.dtype!=bool: values=values.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False})
    stage_pass=bool(values.notna().all() and values.all())
    integrity_rows.append({'stage':stage,'n_checks':len(frame),'all_checks_pass':stage_pass,
                           'source':str(FILES[f'{stage}_integrity'])})
upstream_integrity=pd.DataFrame(integrity_rows)
save_table(upstream_integrity,'02_UPSTREAM_INTEGRITY_LOCKS')
display(upstream_integrity)
if not upstream_integrity.all_checks_pass.all(): raise AssertionError('At least one upstream integrity lock failed.')

## 4. Normalize endpoint and method names

Experiment identifiers remain in provenance tables only. Reader-facing comparisons use methodological names.

In [ ]:
def normalize_endpoint(frame):
    frame=frame.copy(); c=find_col(frame,['endpoint','task'])
    frame['endpoint']=frame[c].astype(str).str.strip().str.lower().str.replace('-','_',regex=False)
    frame['endpoint']=frame.endpoint.replace({'normal_vs_mci':'normal_vs_mci','mci_vs_normal':'normal_vs_mci'})
    return frame

METHOD_NAMES={
 'conventional':'Conventional feature benchmark','selected_deep':'Endpoint-selected deep model',
 'ceednet':'CEEDNet','ssl_attention':'Self-supervised attention encoder','eegnet':'Multiscale EEGNet',
 'cnn_bilstm':'CNN–BiLSTM feature fusion','connectivity_gnn':'Connectivity GNN',
 'reduced_fusion':'Reduced multiview fusion','age_only':'Age only',
 'hybrid_raw':'Hybrid EEG','hybrid_raw_plus_age':'Hybrid EEG + age',
 'regional_raw':'Regional EEG','regional_raw_plus_age':'Regional EEG + age',
 'regional_age_normative':'Age-normative regional EEG','regional_combined':'Combined regional representation'}

def pretty_method(value):
    key=str(value).strip().lower().replace(' ','_').replace('-','_')
    return METHOD_NAMES.get(key,str(value).replace('_',' ').strip().title())

def assert_four(frame,label):
    found=set(frame.endpoint.dropna().unique()); expected=set(EXPECTED_ENDPOINTS)
    if found!=expected: raise AssertionError(f'{label}: endpoint mismatch; missing={expected-found}, extra={found-expected}')

## 5. Development OOF evidence

This is the only layer used for method development and selection. It must not be statistically pooled with official-test or external results.

In [ ]:
development=normalize_endpoint(pd.read_csv(FILES['7BR_summary']))
assert_four(development,'Nested development comparison')
method_col=find_col(development,['method','model','method_name'])
development['method_name']=development[method_col].map(pretty_method)
development['evidence_level']='Nested development OOF'
development['chance_balanced_accuracy']=development.endpoint.map(CHANCE_LEVEL)
save_table(development,'03_DEVELOPMENT_OOF_METHOD_RESULTS')
display(development.head(20))

## 6. Retrospective locked official-test evidence

The conventional benchmark and endpoint-selected deep model are compared on identical official-test recordings with identically computed metrics. These results are retrospective locked-test evidence, not a historically untouched prospective test.

In [ ]:
official=normalize_endpoint(pd.read_csv(FILES['7D2_metrics']))
assert_four(official,'Matched official test')
method_col=find_col(official,['method','model','method_name'])
official['method_name']=official[method_col].map(pretty_method)
official['evidence_level']='Retrospective locked official test'
official['chance_balanced_accuracy']=official.endpoint.map(CHANCE_LEVEL)
official_paired=normalize_endpoint(pd.read_csv(FILES['7D2_paired']))
assert_four(official_paired,'Matched official-test paired comparisons')
save_table(official,'04_MATCHED_OFFICIAL_TEST_RESULTS')
save_table(official_paired,'05_MATCHED_OFFICIAL_TEST_PAIRED_COMPARISONS')
display(official)
display(official_paired)

## 7. Controlled fairness evidence

Classifier comparisons are interpreted only within their prespecified feature or time-series block. They do not establish a universal ranking across incompatible inputs.

In [ ]:
fairness=normalize_endpoint(pd.read_csv(FILES['7D3_metrics']))
assert_four(fairness,'Controlled fairness audit')
method_col=find_col(fairness,['method','model','method_name'])
fairness['method_name']=fairness[method_col].map(pretty_method)
fairness['evidence_level']='Controlled nested development OOF'
fairness_paired=normalize_endpoint(pd.read_csv(FILES['7D3_paired']))
save_table(fairness,'06_CONTROLLED_FAIRNESS_RESULTS')
save_table(fairness_paired,'07_CONTROLLED_FAIRNESS_PAIRED_COMPARISONS')
display(fairness.sort_values(['endpoint','method_name']).head(30))

## 8. Frozen external evidence

AHEPA contributes Normal-vs-AD only. Dryad/P-ADIC contributes all four endpoints when compatible labels are present. External cohorts remain untouched: no refitting, tuning, threshold optimization, or recalibration.

In [ ]:
external=normalize_endpoint(pd.read_csv(FILES['7E_metrics']))
method_col=find_col(external,['method','model','method_name'])
external['method_name']=external[method_col].map(pretty_method)
external['evidence_level']='Frozen external participant-level evaluation'
external['chance_balanced_accuracy']=external.endpoint.map(CHANCE_LEVEL)
external_intervals=normalize_endpoint(pd.read_csv(FILES['7E_intervals']))
external_paired=normalize_endpoint(pd.read_csv(FILES['7E_paired']))
save_table(external,'08_FROZEN_EXTERNAL_RESULTS')
save_table(external_intervals,'09_FROZEN_EXTERNAL_INTERVALS')
save_table(external_paired,'10_FROZEN_EXTERNAL_PAIRED_COMPARISONS')
display(external)
display(external_paired)

## 9. Fold-safe age-confounding and age-normative sensitivity

The age audit uses the same canonical five outer folds across all four endpoints. It compares age-only, EEG-only, EEG-plus-age, regional, and age-normative representations. Holm-adjusted inference is retained exactly as produced upstream.

In [ ]:
age_summary=normalize_endpoint(pd.read_csv(FILES['7F_summary']))
age_deltas=normalize_endpoint(pd.read_csv(FILES['7F_deltas']))
age_distribution=pd.read_csv(FILES['7F_age'])
assert_four(age_summary,'Age sensitivity summary')
assert_four(age_deltas,'Age sensitivity paired deltas')
rep_col=find_col(age_summary,['representation','method','model'])
age_summary['method_name']=age_summary[rep_col].map(pretty_method)
age_summary['evidence_level']='Fold-safe age sensitivity OOF'
save_table(age_summary,'11_AGE_SENSITIVITY_RESULTS')
save_table(age_deltas,'12_AGE_SENSITIVITY_PAIRED_DELTAS')
save_table(age_distribution,'13_AGE_DISTRIBUTION_BY_DIAGNOSIS')
display(age_summary)
display(age_deltas)

## 10. Manuscript-facing evidence-stratified summary

The summary deliberately retains an `evidence_level` column. No pooled ranking is computed across evidence layers.

In [ ]:
def compact_metric_table(frame):
    cols=[]
    for wanted in ['dataset','endpoint','method_name','evidence_level','n','accuracy','balanced_accuracy','macro_f1','mcc','roc_auc','macro_ovr_roc_auc','chance_balanced_accuracy']:
        if wanted in frame.columns: cols.append(wanted)
    return frame[cols].copy()

publication_summary=pd.concat([
    compact_metric_table(development),compact_metric_table(official),compact_metric_table(external)
],ignore_index=True,sort=False)
publication_summary['endpoint_label']=publication_summary.endpoint.map(ENDPOINT_LABELS)
publication_summary=publication_summary.sort_values(['evidence_level','endpoint','method_name']).reset_index(drop=True)
save_table(publication_summary,'14_EVIDENCE_STRATIFIED_PUBLICATION_SUMMARY')
display(publication_summary)

## 11. Publication figures

Figures use balanced accuracy because it has explicit chance levels of 0.50 for binary endpoints and 0.333 for the three-class endpoint. Panels are separated by evidence design.

In [ ]:
def ba_col(frame): return find_col(frame,['balanced_accuracy','ba'])

fig,axes=plt.subplots(1,2,figsize=(13,5),sharey=True)
for ax,(title,frame) in zip(axes,[('Nested development OOF',development),('Retrospective locked official test',official)]):
    b=ba_col(frame); plot=frame.copy(); plot['endpoint_label']=plot.endpoint.map(ENDPOINT_LABELS)
    sns.pointplot(data=plot,x='endpoint_label',y=b,hue='method_name',dodge=.35,markers='o',linestyles='-',errorbar=None,ax=ax)
    ax.set_title(title,fontweight='bold'); ax.set_xlabel(''); ax.set_ylabel('Balanced accuracy' if ax is axes[0] else '')
    ax.set_ylim(0.25,1.0); ax.tick_params(axis='x',rotation=25); ax.legend(fontsize=7,title='Method',loc='best')
fig.suptitle('Internal evidence by endpoint and evaluation design',fontweight='bold'); fig.tight_layout()
for ext in ('pdf','png'):
    fig.savefig(FIGURES/f'Figure_01_Internal_Evidence_Stratified_Comparison.{ext}',dpi=600 if ext=='png' else None,bbox_inches='tight',facecolor='white')
plt.show()

dataset_col=find_col(external,['dataset','cohort'])
fig,ax=plt.subplots(figsize=(11,5.5)); plot=external.copy(); plot['panel']=plot[dataset_col].astype(str)+' — '+plot.endpoint.map(ENDPOINT_LABELS)
sns.barplot(data=plot,x='panel',y=ba_col(plot),hue='method_name',ax=ax,errorbar=None)
for i,label in enumerate(plot['panel'].drop_duplicates()):
    endpoint=plot.loc[plot.panel==label,'endpoint'].iloc[0]; ax.hlines(CHANCE_LEVEL[endpoint],i-.42,i+.42,color='black',linestyle='--',linewidth=1)
ax.set_ylim(0,1); ax.set_xlabel('External dataset and endpoint'); ax.set_ylabel('Balanced accuracy')
ax.set_title('Frozen external evaluation (dashed segments show chance)',fontweight='bold'); ax.tick_params(axis='x',rotation=30)
ax.legend(title='Method',fontsize=8); fig.tight_layout()
for ext in ('pdf','png'):
    fig.savefig(FIGURES/f'Figure_02_Frozen_External_Performance.{ext}',dpi=600 if ext=='png' else None,bbox_inches='tight',facecolor='white')
plt.show()

fig,ax=plt.subplots(figsize=(12,5.5)); plot=age_summary.copy(); plot['endpoint_label']=plot.endpoint.map(ENDPOINT_LABELS)
sns.pointplot(data=plot,x='endpoint_label',y=ba_col(plot),hue='method_name',dodge=.45,markers='o',linestyles='-',errorbar=None,ax=ax)
ax.set_ylim(0.3,0.9); ax.set_xlabel(''); ax.set_ylabel('Balanced accuracy'); ax.set_title('Fold-safe age sensitivity across four endpoints',fontweight='bold')
ax.tick_params(axis='x',rotation=20); ax.legend(title='Representation',fontsize=7,ncol=2); fig.tight_layout()
for ext in ('pdf','png'):
    fig.savefig(FIGURES/f'Figure_03_Four_Endpoint_Age_Sensitivity.{ext}',dpi=600 if ext=='png' else None,bbox_inches='tight',facecolor='white')
plt.show()

## 12. Manuscript evidence map

This table defines which source may support each manuscript claim and prevents obsolete results from re-entering the paper.

In [ ]:
evidence_map=pd.DataFrame([
 {'manuscript_topic':'Conventional model development','evidence_design':'Nested development OOF','source_table':'03_DEVELOPMENT_OOF_METHOD_RESULTS.csv','claim_boundary':'Development comparison only; not external generalization'},
 {'manuscript_topic':'Final internal comparison','evidence_design':'Retrospective locked official test','source_table':'04_MATCHED_OFFICIAL_TEST_RESULTS.csv','claim_boundary':'Identical recordings and metrics; retrospective lock'},
 {'manuscript_topic':'Fairness of adopted methods','evidence_design':'Controlled nested development OOF','source_table':'06_CONTROLLED_FAIRNESS_RESULTS.csv','claim_boundary':'Compare only within the same input block'},
 {'manuscript_topic':'Transportability','evidence_design':'Frozen participant-level external evaluation','source_table':'08_FROZEN_EXTERNAL_RESULTS.csv','claim_boundary':'No external tuning; dataset-specific transportability'},
 {'manuscript_topic':'Age confounding and sensitivity','evidence_design':'Fold-safe development OOF','source_table':'11_AGE_SENSITIVITY_RESULTS.csv','claim_boundary':'Age contribution and sensitivity; no causal interpretation'},
])
save_table(evidence_map,'15_MANUSCRIPT_EVIDENCE_MAP'); display(evidence_map)

## 13. Final reproducibility and publication lock

The final decision is written only if all endpoints, source integrity locks, evidence boundaries, and finite metric checks pass.

In [ ]:
checks=[]
def add_check(name,value,detail=''):
    checks.append({'check':name,'pass':bool(value),'detail':detail})

add_check('all_upstream_integrity_locks_pass',upstream_integrity.all_checks_pass.all())
add_check('development_has_four_endpoints',set(development.endpoint)==set(EXPECTED_ENDPOINTS))
add_check('official_test_has_four_endpoints',set(official.endpoint)==set(EXPECTED_ENDPOINTS))
add_check('age_audit_has_four_endpoints',set(age_summary.endpoint)==set(EXPECTED_ENDPOINTS))
add_check('binary_chance_is_0_5',all(CHANCE_LEVEL[e]==.5 for e in EXPECTED_ENDPOINTS[:3]))
add_check('three_class_chance_is_one_third',np.isclose(CHANCE_LEVEL['three_class'],1/3))
add_check('evidence_levels_not_pooled',publication_summary.evidence_level.nunique()>=3)
for label,frame in [('development',development),('official',official),('external',external),('age',age_summary)]:
    b=ba_col(frame); values=pd.to_numeric(frame[b],errors='coerce')
    add_check(f'{label}_balanced_accuracy_finite',np.isfinite(values).all())
    add_check(f'{label}_balanced_accuracy_bounded',values.between(0,1).all())

final_checks=pd.DataFrame(checks); save_table(final_checks,'16_FINAL_INTEGRITY_CHECKS'); display(final_checks)
if not final_checks['pass'].all(): raise AssertionError('P1-E16.7G final publication-lock integrity failure.')

outputs=[]
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file(): outputs.append({'path':str(path),'sha256':sha256_file(path),'size_bytes':path.stat().st_size})
output_manifest=pd.DataFrame(outputs); save_table(output_manifest,'17_OUTPUT_MANIFEST')

decision={
 'experiment':'P1-E16.7G',
 'status':'PASSED — FOUR-ENDPOINT STATISTICAL, REPRODUCIBILITY AND PUBLICATION LOCKED',
 'completed_utc':datetime.now(timezone.utc).isoformat(),
 'endpoints':EXPECTED_ENDPOINTS,
 'evidence_levels':['nested development OOF','retrospective locked official test','frozen external participant-level evaluation'],
 'primary_internal_benchmark':'Conventional feature benchmark',
 'secondary_evidence':'Endpoint-selected deep and controlled adaptation methods',
 'external_claim':'Dataset-specific frozen transportability; no clinical-readiness or universal-generalization claim',
 'new_training_performed':False,
 'seed':SEED,
}
(OUTPUT_ROOT/'P1_E16_7G_FINAL_DECISION.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
print(decision['status'])